# 03 - Feature Engineering
Date features, duration, parsing structured columns, tag / rating / related-talk features.

**Input:** `/content/processed/02_clean_df.pkl`  
**Output:** `/content/processed/03_featured_df.pkl`

Next notebook: `04_Data_Visualization.ipynb`

## 1. Load cleaned data

In [ ]:
import pandas as pd

clean_df = pd.read_pickle("/content/processed/02_clean_df.pkl")
print(clean_df.shape)

## 2. Date features

In [ ]:
clean_df["film_datetime"] = pd.to_datetime(
    clean_df["film_date"],
    unit="s",
    errors="coerce",
    utc=True
)

clean_df["published_datetime"] = pd.to_datetime(
    clean_df["published_date"],
    unit="s",
    errors="coerce",
    utc=True
)

# Create additional features
clean_df["film_year"] = clean_df["film_datetime"].dt.year
clean_df["film_month"] = clean_df["film_datetime"].dt.month
clean_df["published_year"] = clean_df["published_datetime"].dt.year

print(clean_df[
    ["film_date", "film_datetime",
     "published_date", "published_datetime",
     "film_year", "film_month", "published_year"]
].head())

In [ ]:
# This measures how long it took for a talk to be published after recording.


clean_df["publication_delay_days"] = (
    clean_df["published_datetime"] -
    clean_df["film_datetime"]
).dt.total_seconds() / 86400

print(clean_df[
    ["title", "publication_delay_days"]
].head())

## 3. Duration in minutes

In [ ]:
# Convert duration from seconds to minutes


clean_df["duration_minutes"] = clean_df["duration"] / 60

print(clean_df[
    ["title", "duration", "duration_minutes"]
].head())

## 4. Parse list / dictionary columns (`tags`, `ratings`, `related_talks`)

In [ ]:
import ast

# These columns contain structured information stored as text.
# We should convert the text into Python lists or dictionaries before using the information.

# For example, a tags value might look like this:

# "['education', 'technology', 'science']"



def parse_list(value):
    if isinstance(value, list):
        return value

    if not isinstance(value, str) or not value.strip():
        return []

    try:
        result = ast.literal_eval(value)

        if isinstance(result, list):
            return result

        return []

    except (ValueError, SyntaxError):
        return []


for col in ["tags", "ratings", "related_talks"]:
    clean_df[col] = clean_df[col].apply(parse_list)

print("Tags example:")
print(clean_df["tags"].iloc[0])

print("\nRatings example:")
print(clean_df["ratings"].iloc[0])

print("\nRelated talks example:")
print(clean_df["related_talks"].iloc[0])

## 5. Tag features

In [ ]:
# Create features from tags


clean_df["tags_text"] = clean_df["tags"].apply(
    lambda tags: " ".join(
        str(tag).strip().lower()
        for tag in tags
        if isinstance(tag, str) and tag.strip()
    )
)

clean_df["tag_count"] = clean_df["tags"].apply(len)

print(clean_df[
    ["title", "tags", "tags_text", "tag_count"]
].head())

## 6. Rating features

In [ ]:
# This feature counts rating categories, not the number of individual audience votes.
# We can calculate rating totals and category-specific features
# after confirming the exact dictionary structure.


clean_df["rating_category_count"] = clean_df["ratings"].apply(
    lambda ratings: sum(
        1 for item in ratings if isinstance(item, dict)
    )
)

print(clean_df[
    ["title", "ratings", "rating_category_count"]
].head())

## 7. Related talk features

In [ ]:
# Create features from related talks


clean_df["related_talk_count"] = clean_df["related_talks"].apply(
    len
)

print(clean_df[
    ["title", "related_talks", "related_talk_count"]
].head())

## 8. Save output for the next notebook

In [ ]:
clean_df.to_pickle("/content/processed/03_featured_df.pkl")
print("Saved:", clean_df.shape)